<a href="https://colab.research.google.com/github/herrypu/HP_LLM4CD_S6SP/blob/main/chipchat_exampleB_i2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM4ChipDesign HW1 Chipchat
## Example B (Example 8: Traffic Light)


In [1]:
API_KEY="sk-4b4b50b8153147759b5530268e51a1f7"
MAX_TOKENS=4096
TEMPERATURE=0.7

### 1. Setting up

In [2]:
### Installing dependencies
!pip install openai

!apt-get update
!apt-get install -y iverilog

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
iverilog is already the newest version (11.0-1.1).
0 upgraded, 0 newly installed, 0 to remove 

In [3]:
def ask_deepseek(prompt):
    from openai import OpenAI
    client = OpenAI(
        api_key = API_KEY,
        base_url = "https://api.deepseek.com"
    )
    completion = client.chat.completions.create(
        model="deepseek-reasoner",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE
    )
    content = completion.choices[0].message.content

    print("-" * 30 + " LLM Response " + "-" * 30)
    print(content)
    print("-" * 74)

    return content

### 2. Verilog Generation

In [4]:
!mkdir -p traffic_light/
!cd traffic_light/ && curl -O https://raw.githubusercontent.com/FCHXWH823/LLM4ChipDesign/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/traffic_light_tb.v

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  2018  100  2018    0     0   5098      0 --:--:-- --:--:-- --:--:--  5108


In [5]:
verilog_generation_prompt = '''
I am trying to create a Verilog model for a traffic light state machine. It must meet the following specifications:
    - Inputs:
        - Clock
        - Active-low reset
        - Enable
    - Outputs:
        - Red
        - Yellow
        - Green

The state machine should reset to a red light, change from red to green after 32 clock cycles, change from green to yellow after 20 clock cycles, and then change from yellow to red after 7 clock cycles.

How would I write a design that meets these specifications?
'''

In [6]:
ask_deepseek(verilog_generation_prompt)

------------------------------ LLM Response ------------------------------
To create a Verilog model for a traffic light state machine meeting your specifications, you can follow the design below. This implementation uses a finite state machine (FSM) with three states (RED, GREEN, YELLOW) and a counter to control the timing for each state.

### Verilog Code:
```verilog
module traffic_light (
    input wire clk,        // Clock input
    input wire reset_n,    // Active-low reset
    input wire enable,     // Enable signal
    output reg red,        // Red light output
    output reg yellow,     // Yellow light output
    output reg green       // Green light output
);

    // Define states
    localparam [1:0] RED_STATE    = 2'b00;
    localparam [1:0] GREEN_STATE  = 2'b01;
    localparam [1:0] YELLOW_STATE = 2'b10;

    // Internal state and counter registers
    reg [1:0] current_state, next_state;
    reg [5:0] counter; // 6-bit counter (max 64 cycles)

    // Sequential logic for s

"To create a Verilog model for a traffic light state machine meeting your specifications, you can follow the design below. This implementation uses a finite state machine (FSM) with three states (RED, GREEN, YELLOW) and a counter to control the timing for each state.\n\n### Verilog Code:\n```verilog\nmodule traffic_light (\n    input wire clk,        // Clock input\n    input wire reset_n,    // Active-low reset\n    input wire enable,     // Enable signal\n    output reg red,        // Red light output\n    output reg yellow,     // Yellow light output\n    output reg green       // Green light output\n);\n\n    // Define states\n    localparam [1:0] RED_STATE    = 2'b00;\n    localparam [1:0] GREEN_STATE  = 2'b01;\n    localparam [1:0] YELLOW_STATE = 2'b10;\n\n    // Internal state and counter registers\n    reg [1:0] current_state, next_state;\n    reg [5:0] counter; // 6-bit counter (max 64 cycles)\n\n    // Sequential logic for state and counter updates\n    always @(posedge clk o

### 3. Extraction and Verification

In [9]:
output_verilog_code = '''
module traffic_light (
    input wire clk,        // Clock input
    input wire reset_n,    // Active-low reset
    input wire enable,     // Enable signal
    output reg red,        // Red light output
    output reg yellow,     // Yellow light output
    output reg green       // Green light output
);

    // Define states
    localparam [1:0] RED_STATE    = 2'b00;
    localparam [1:0] GREEN_STATE  = 2'b01;
    localparam [1:0] YELLOW_STATE = 2'b10;

    // Internal state and counter registers
    reg [1:0] current_state, next_state;
    reg [5:0] counter; // 6-bit counter (max 64 cycles)

    // Sequential logic for state and counter updates
    always @(posedge clk or negedge reset_n) begin
        if (!reset_n) begin
            current_state <= RED_STATE;
            counter <= 6'd0;
        end else if (enable) begin
            current_state <= next_state;
            // Reset counter on state change or increment if staying in the same state
            if (next_state != current_state) begin
                counter <= 6'd0;
            end else begin
                counter <= counter + 1;
            end
        end
    end

    // Combinational logic for next state and outputs
    always @(*) begin
        // Default outputs
        red    = 1'b0;
        yellow = 1'b0;
        green  = 1'b0;

        case (current_state)
            RED_STATE: begin
                red = 1'b1;
                if (counter >= 6'd31) begin // 32 cycles (0 to 31)
                    next_state = GREEN_STATE;
                end else begin
                    next_state = RED_STATE;
                end
            end

            GREEN_STATE: begin
                green = 1'b1;
                if (counter >= 6'd19) begin // 20 cycles (0 to 19)
                    next_state = YELLOW_STATE;
                end else begin
                    next_state = GREEN_STATE;
                end
            end

            YELLOW_STATE: begin
                yellow = 1'b1;
                if (counter >= 6'd6) begin // 7 cycles (0 to 6)
                    next_state = RED_STATE;
                end else begin
                    next_state = YELLOW_STATE;
                end
            end

            default: begin
                next_state = RED_STATE;
            end
        endcase
    end

endmodule
'''
filename = "traffic_light/traffic_light.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [10]:
!cd traffic_light/ && iverilog -o traffic_light.vvp traffic_light.v traffic_light_tb.v && vvp traffic_light.vvp

traffic_light_tb.v:32: syntax error
